In [1]:
import pandas as pd
from matplotlib import pyplot as plt 
import seaborn as sns
from ast import literal_eval
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import joblib
import numpy as np
from surprise import Dataset, Reader, SVD
from sklearn.preprocessing import StandardScaler
from scipy.sparse import csr_matrix
from scipy.sparse import hstack


In [2]:
df1 = pd.read_csv("../Data/tmdb_5000_movies.csv")
df2 = pd.read_csv("../Data/tmdb_5000_credits.csv")
df3 = df1.merge(df2, on='title')
deleted_columns = [
    'homepage', 'original_language', 'production_countries', 'production_companies', 'spoken_languages', 'status', 'tagline',
    'movie_id', 'runtime', 'release_date', 'popularity', 'original_title'
    ]
df3 = df3.drop(columns=deleted_columns, errors='ignore')
df3 = df3.dropna().reset_index()

def extract(text):
    str = literal_eval(text)
    list = []
    for i in str:
        if i['name']:
            list.append(i['name'].replace(" ", ""))
    return list

def ext_dir(text):
    str = literal_eval(text)
    list = []
    for i in str:
        if i['job'] == 'Director':
            list.append(i['name'].replace(" ", ""))
    return list

ps = PorterStemmer()

def stem(t):
    list = []
    for i in t:
        list.append(ps.stem(i))
    return list

df3['genres'] = df3['genres'].apply(extract)
df3['keywords'] = df3['keywords'].apply(extract)
df3['cast'] = df3['cast'].apply(extract).apply(lambda x: x[:5])
df3['crew']= df3['crew'].apply(ext_dir)
df3['overview'] = df3['overview'].apply(lambda x : x.split())
df3['tags'] = (df3["genres"] + df3["keywords"] + df3["overview"] + df3["cast"] + df3["crew"])

df = df3[['id', 'title', 'tags', 'vote_average']].copy()
df['tags'] = df['tags'].apply(stem).apply(lambda x: " ".join(x))

model = CountVectorizer(max_features=5000, stop_words='english')
vectors = model.fit_transform(df['tags'])

scaler = StandardScaler()
scales = scaler.fit_transform(df3[["budget", "revenue", 'vote_average']])
scales = csr_matrix(scales, dtype="int32")

paras = hstack([vectors, scales])
similarity = cosine_similarity(paras, dense_output=True)

joblib.dump(similarity, "Similarity.pkl", compress=3)
joblib.dump(df, "Movies_Data.pkl", compress=3)

['Movies_Data.pkl']

In [4]:
df3.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4806 entries, 0 to 4805
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   index         4806 non-null   int64  
 1   budget        4806 non-null   int64  
 2   genres        4806 non-null   object 
 3   id            4806 non-null   int64  
 4   keywords      4806 non-null   object 
 5   overview      4806 non-null   object 
 6   revenue       4806 non-null   int64  
 7   title         4806 non-null   object 
 8   vote_average  4806 non-null   float64
 9   vote_count    4806 non-null   int64  
 10  cast          4806 non-null   object 
 11  crew          4806 non-null   object 
 12  tags          4806 non-null   object 
dtypes: float64(1), int64(5), object(7)
memory usage: 488.2+ KB


In [4]:
# CREATING RATING DATASET
# ==========================================================
user_id = np.arange(1, 30000)
ratings = []

for idx, row in df1.iterrows():
    basic_rating = row['vote_average'] / 2
    movie = row['title']
    size = row['vote_count']
    users = np.random.choice(user_id, size=size, replace=False)
    for user in users:
        rating_variance = np.random.uniform(-0.5, 0.5)
        rating = basic_rating + rating_variance
        final_rating = float(np.clip(rating, 1.0, 5.0))
        final_rating = np.round(final_rating, 1)

        ratings.append({
            "User Id": user,
            "Movie Title": movie,
            "Rating": final_rating
        })
        
ratings_set = pd.DataFrame(ratings)
ratings_set_max = ratings_set['Rating'].max()
ratings_set_min = ratings_set['Rating'].min()
reader = Reader(rating_scale=(ratings_set_min, ratings_set_max))
data_surprise = Dataset.load_from_df(ratings_set, reader)
trainset = data_surprise.build_full_trainset()

svd_model = SVD(n_factors=50, random_state=42)
svd_model.fit(trainset)
joblib.dump(svd_model, "Model_SVD.pkl", compress=3)
joblib.dump(ratings_set, "User_data.pkl", compress=3)

['User_data.pkl']

In [24]:
similarity.max()

np.float64(1.0000000000000018)